# Students' Dropout and Academic success Analysis

**Цель**: Разработать систему раннего выявления студентов группы риска (риск отчисления) на ранних этапах обучения.


In [1]:
import pandas as pd

In [2]:
# Загрузка данных


In [3]:
df.head(10)

,Unnamed: 0,marital_status,app_mode,app_order,course,attendance,prev_edu,prev_grade,nationality,mom_edu,...,sem2_evaluations,sem2_passed,sem2_grade,sem2_no_grade,unemployment,inflation,gdp,target,gender_label,international_label
0,0,1,17,5,171,1,1,122.0,1,19,...,0,0,0.000000,0,10.8,1.4,1.74,Dropout,Male,Local
1,1,1,15,1,9254,1,1,160.0,1,1,...,6,6,13.666667,0,13.9,-0.3,0.79,Graduate,Male,Local
2,2,1,1,5,9070,1,1,122.0,1,37,...,0,0,0.000000,0,10.8,1.4,1.74,Dropout,Male,Local
3,3,1,17,2,9773,1,1,122.0,1,38,...,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate,Female,Local
4,4,2,39,1,8014,0,1,100.0,1,37,...,6,6,13.000000,0,13.9,-0.3,0.79,Graduate,Female,Local
5,5,2,39,1,9991,0,19,133.1,1,37,...,17,5,11.500000,5,16.2,0.3,-0.92,Graduate,Male,Local
6,6,1,1,1,9500,1,1,142.0,1,19,...,8,8,14.345000,0,15.5,2.8,-4.06,Graduate,Female,Local
7,7,1,18,4,9254,1,1,119.0,1,37,...,5,0,0.000000,0,15.5,2.8,-4.06,Dropout,Male,Local
8,8,1,1,3,9238,1,1,137.0,62,1,...,7,6,14.142857,0,16.2,0.3,-0.92,Graduate,Female,International
9,9,1,1,1,9238,1,1,138.0,1,1,...,14,2,13.500000,0,8.9,1.4,3.51,Dropout,Female,Local


### Анализ оттока (Dropout Analysis)

**Гипотезы:**
- Студенты с низким средним баллом в первом семестре чаще отчисляются
- Студенты с задолженностью по оплате (debtor=1) имеют в 3 раза выше риск отчисления
- Студенты вечерней формы обучения реже отчисляются 
- Более взрослые студенты реже отчисляются(больше 25 лет)
- Балл при поступлении влияет на вероятность отчисления
- Наличие стипендии положительно влияет на успешное окончание обучения

Для проверки гипотез будут использоваться данные об учениках, которые выпустились, и об учениках, которые отчислились. Ученики, продолжающие обучение, не будут включены в анализ, поскольку их итоговый статус еще не определен и они могут исказить статистический анализ.

## Гипотеза 1: Студенты с низким средним баллом в первом семестре чаще отчисляются
**За низкий балл возьмем балл, ниже медианного**

In [4]:
import numpy as np
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Фильтрация данных
# Берем только Graduate и Dropout 
df_final = df[df['target'].isin(['Graduate', 'Dropout'])].copy()

print(f"Всего студентов в анализе: {len(df_final)}")
print(f"Выпускников (Graduate): {(df_final['target'] == 'Graduate').sum()}")
print(f"Отчисленных (Dropout): {(df_final['target'] == 'Dropout').sum()}")
print(f"Доля отчисленных: {(df_final['target'] == 'Dropout').mean()*100:.1f}%")

Всего студентов в анализе: 3630
Выпускников (Graduate): 2209
Отчисленных (Dropout): 1421
Доля отчисленных: 39.1%


In [6]:
# Рассчитываем медиану баллов 1 семестра
median_grade = df_final['sem1_grade'].median()
print(f"Медианный балл 1 семестра: {median_grade:.2f}")

Медианный балл 1 семестра: 12.34


In [7]:
# Создаем бинарную переменную: низкий балл (ниже медианы) vs высокий балл
df_final['low_grade'] = df_final['sem1_grade'] < median_grade
df_final['low_grade_label'] = df_final['low_grade'].map({True: 'Низкий балл', False: 'Высокий балл'})

In [8]:
# Смотрим распределение
df_final.groupby('low_grade_label').size()

low_grade_label
Высокий балл    1815
Низкий балл     1815
dtype: int64

**Для проверки гипотезы используем статистический тест Хи-квадрат.**

In [9]:
# Создаем таблицу сопряженности
h1_contingency = pd.crosstab(df_final['low_grade_label'], df_final['target'])
print('Таблица сопряженности')
h1_contingency

Таблица сопряженности


target,Dropout,Graduate
low_grade_label,,
Высокий балл,312,1503
Низкий балл,1109,706


In [10]:
# Рассчитываем проценты
h1_percent = h1_contingency.div(h1_contingency.sum(axis=1), axis=0) * 100
print('Проценты (по строкам):')
h1_percent.round(1)

Проценты (по строкам):


target,Dropout,Graduate
low_grade_label,,
Высокий балл,17.2,82.8
Низкий балл,61.1,38.9


In [11]:
# Функция для проведения теста Хи-квадрат и рассчета Odds Ratio (отношение шансов)

def calculate_chi2_or(contingency_table, ctrl_idx=0, exp_idx=1):
    """
    Предполагает, что таблица имеет структуру:
    - row 0: контрольная группа
    - row 1: экспериментальная группа
    - columns: ['Dropout', 'Graduate']

    Контрольная группа - группа, в которой предположительно реже отчисляются
    Экспериментальная группа - группа, в которой предположительно чаще отчисляются
    """
    # Тест хи-квадрат для проверки связи
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    print('Результаты теста:')
    print(f"хи-квадрат = {chi2:.3f}")
    print(f"p-value = {p_value:.8f}")
    print(f"Степени свободы (df) = {dof}")

    # Интерпретация p-value
    if p_value < 0.05:
        print('Статистически значимо')
        print('Есть связь между низким баллом и отчислением (p < 0.05)')
        print('------------------------')
    else:
        print('Не значимо')
        print('Нет статистической связи (p ≥ 0.05)')
        print('------------------------')
        
        

    
    # Контрольная группа 
    ctrl_dropout = contingency_table.iloc[ctrl_idx, contingency_table.columns.get_loc('Dropout')]
    ctrl_graduate = contingency_table.iloc[ctrl_idx, contingency_table.columns.get_loc('Graduate')]
    
    # Экспериментальная группа 
    exp_dropout = contingency_table.iloc[exp_idx, contingency_table.columns.get_loc('Dropout')]
    exp_graduate = contingency_table.iloc[exp_idx, contingency_table.columns.get_loc('Graduate')]
    
    
    # Odds для контрольной группы
    odds_ctrl = ctrl_dropout / ctrl_graduate if ctrl_graduate > 0 else np.inf
    
    # Odds для экспериментальной группы
    odds_exp = exp_dropout / exp_graduate if exp_graduate > 0 else np.inf
    
    # Odds Ratio
    or_value = odds_exp / odds_ctrl if odds_ctrl > 0 else np.inf
    
     # Рассчитываем общее количество студентов в каждой группе
    exp_total = exp_dropout + exp_graduate
    ctrl_total = ctrl_dropout + ctrl_graduate
    
    # Процент отчисляемости для каждой группы
    exp_dropout_pct = (exp_dropout / exp_total * 100) if exp_total > 0 else 0
    ctrl_dropout_pct = (ctrl_dropout / ctrl_total * 100) if ctrl_total > 0 else 0
    
    print('Отношение шансов:')
    print(f"Шанс отчисления в экспериментальной группе: {odds_exp:.3f} (1:{1/odds_exp:.1f})")
    print(f"Шанс отчисления в контрольной группе: {odds_ctrl:.3f} (1:{1/odds_ctrl:.1f})")
    print(f"Odds Ratio = {or_value:.2f}")
    print('------------------------')
    
    if or_value > 1:
        print(f"Студенты в экспериментальной группе отчисляются в {or_value:.1f} раз чаще.")
        print('Гипотеза подтвердилась.')
        print('------------------------')
    elif or_value < 1:
        print(f"Студенты в экспериментальной группе отчисляются реже (в {1/or_value:.1f} раз).")
        print('Гипотеза опровергнута.')
    else:
        print('Нет разницы в шансах отчисления')
        print('------------------------')
    
    print('Отчисляемость студентов:')
    print(f"Экспериментальная группа: {exp_dropout_pct:.1f}% ({exp_dropout}/{exp_total})")
    print(f"Контрольная группа: {ctrl_dropout_pct:.1f}% ({ctrl_dropout}/{ctrl_total})")
    
    return {
        'odds_exp': odds_exp,
        'odds_ctrl': odds_ctrl,
        'or': or_value
    }

In [12]:
# Тест хи-квадрат для проверки связи
# Рассчет Odds Ratio 
h1_result = calculate_chi2_or(h1_contingency)

Результаты теста:
хи-квадрат = 732.728
p-value = 0.00000000
Степени свободы (df) = 1
Статистически значимо
Есть связь между низким баллом и отчислением (p < 0.05)
------------------------
Отношение шансов:
Шанс отчисления в экспериментальной группе: 1.571 (1:0.6)
Шанс отчисления в контрольной группе: 0.208 (1:4.8)
Odds Ratio = 7.57
------------------------
Студенты в экспериментальной группе отчисляются в 7.6 раз чаще.
Гипотеза подтвердилась.
------------------------
Отчисляемость студентов:
Экспериментальная группа: 61.1% (1109/1815)
Контрольная группа: 17.2% (312/1815)


**Гипотеза 1: Влияние среднего балла на отчисление**

**Подтверждена**

*Результаты:*
- Отчисляемость студентов с низким баллом: 61.1%
- Отчисляемость студентов с высоким баллом: 17.2%
- Студенты с низким баллом отчисляются в **7.6 раз чаще** (OR=7.6, p<0.05)

*Вывод:*

Студенты с баллом ниже медианы отчисляются в 7.6 раз чаще.
Низкий балл 1-го семестра — сильный предиктор отчисления.

*Возможные причины*:
1. Недостаток базовых знаний/навыков
2. Отсутствие интереса к специальности или учёбе в целом
3. Работа, семейные обстоятельства, влияющие на учёбу

*Рекомендация:*
1. Раннее выявление групп риска — внедрить систему мониторинга успеваемости после 1-го семестра с индивидуальными образовательными траекториями для отстающих

2. Академическая поддержка — создать обязательные коррекционные курсы и менторскую программу для студентов с низким баллом

3. Профориентация и адаптация — усилить довузовскую подготовку и вводные курсы для плавного перехода к университетским требованиям

## Гипотеза 2: Студенты с задолженностью по оплате (debtor=1) имеют выше риск отчисления
**Для проверки гипотезы используем статистический тест Хи-квадрат.**

In [13]:
# Смотрим распределение
df_final.groupby('debtor').size()

debtor
0    3217
1     413
dtype: int64

In [14]:
# Создаем таблицу сопряженности
h2_contingency = pd.crosstab(df_final['debtor'], df_final['target'])
print("Таблица сопряженности")
h2_contingency

Таблица сопряженности


target,Dropout,Graduate
debtor,,
0,1109,2108
1,312,101


In [15]:
# Рассчитываем проценты
h2_percent = h2_contingency.div(h2_contingency.sum(axis=1), axis=0) * 100
print('Проценты (по строкам):')
h2_percent.round(1)

Проценты (по строкам):


target,Dropout,Graduate
debtor,,
0,34.5,65.5
1,75.5,24.5


In [16]:
# Тест хи-квадрат для проверки связи
# Рассчет Odds Ratio 

h2_result = calculate_chi2_or(h2_contingency)

Результаты теста:
хи-квадрат = 257.460
p-value = 0.00000000
Степени свободы (df) = 1
Статистически значимо
Есть связь между низким баллом и отчислением (p < 0.05)
------------------------
Отношение шансов:
Шанс отчисления в экспериментальной группе: 3.089 (1:0.3)
Шанс отчисления в контрольной группе: 0.526 (1:1.9)
Odds Ratio = 5.87
------------------------
Студенты в экспериментальной группе отчисляются в 5.9 раз чаще.
Гипотеза подтвердилась.
------------------------
Отчисляемость студентов:
Экспериментальная группа: 75.5% (312/413)
Контрольная группа: 34.5% (1109/3217)


#### Гипотеза 2: Влияние финансового положения на отчисление

**Подтверждена**

*Результаты*
- Отчисляемость должников: 75.5%
- Отчисляемость не должников: 34.5%
- Должники отчисляются в **5.9 раз чаще** (OR=5.9, p<0.05)

*Вывод:* Студенты с задолженностью по оплате отчисляются чаще в 5.9 раз чаще студентов без долга. 

*Возможные причины:*
1. Финансовый стресс может косвенно влиять на успеваемость
2. Административные барьеры — студенты с долгами могут быть ограничены в доступе к ресурсам
3. Мотивационный фактор — финансовые проблемы снижают фокус на учёбе

*Рекомендации:*
1. Гибкая система оплаты — ввести рассрочку, ступенчатые платежи, корреляцию стоимости с успеваемостью

2. Целевая финансовая поддержка — расширить программу скидок и грантов для студентов с хорошей успеваемостью, но сложным финансовым положением

3. Финансовое консультирование — создать службу помощи в планировании бюджета и поиске подработок/стажировок без ущерба учёбе

## Гипотеза 3: Студенты вечерней формы обучения реже отчисляются
**Для проверки гипотезы используем статистический тест Хи-квадрат.**

In [17]:
# Смотрим распределение
df_final.groupby('attendance').size()

attendance
0     408
1    3222
dtype: int64

In [18]:
# Создаем таблицу сопряженности
h3_contingency = pd.crosstab(df_final['attendance'], df_final['target'])
print("Таблица сопряженности")
h3_contingency

Таблица сопряженности


target,Dropout,Graduate
attendance,,
0,207,201
1,1214,2008


In [19]:
# Рассчитываем проценты
h3_percent = h3_contingency.div(h3_contingency.sum(axis=1), axis=0) * 100
print('Проценты (по строкам):')
h3_percent.round(1)

Проценты (по строкам):


target,Dropout,Graduate
attendance,,
0,50.7,49.3
1,37.7,62.3


In [20]:
# Тест хи-квадрат для проверки связи
# Рассчет Odds Ratio 
h3_result = calculate_chi2_or(h3_contingency)

Результаты теста:
хи-квадрат = 25.371
p-value = 0.00000047
Степени свободы (df) = 1
Статистически значимо
Есть связь между низким баллом и отчислением (p < 0.05)
------------------------
Отношение шансов:
Шанс отчисления в экспериментальной группе: 0.605 (1:1.7)
Шанс отчисления в контрольной группе: 1.030 (1:1.0)
Odds Ratio = 0.59
------------------------
Студенты в экспериментальной группе отчисляются реже (в 1.7 раз).
Гипотеза опровергнута.
Отчисляемость студентов:
Экспериментальная группа: 37.7% (1214/3222)
Контрольная группа: 50.7% (207/408)


#### Гипотеза 3: Форма обучения и отчисление

**Опровергнута** (обратный эффект)

*Результаты:*
- Дневная форма: 37.7% отчисленных
- Вечерняя форма: 50.7% отчисленных  
- Вечерние отчисляются в **1.7 раза чаще** (OR=0.59, p<0.05)

*Вывод:* Вечерняя форма обучения ассоциирована с *более высоким* риском отчисления.

*Возможные причины:*
1. Перегрузка от совмещения работы и учёбы
2. Меньше времени на подготовку
3. Ограниченный доступ к ресурсам вечером

*Рекомендации:* 
1. Специализированная адаптация — отдельная программа вводных курсов и методических материалов, оптимизированных для самостоятельной работы

2. Технологическая поддержка — полная цифровизация учебных материалов, запись лекций и расширенный доступ к электронным ресурсам


## Гипотеза 4: Более взрослые студенты реже отчисляются(больше 25 лет)
**Для проверки гипотезы используем статистический тест Хи-квадрат.**

In [21]:
# Создадим бинарную переменную, определяющую, является ли студент молодым
df_final['young_student'] = (df_final['age'] <= 25).astype(int)

In [22]:
# Смотрим распределение
df_final.groupby('young_student').size()

young_student
0     904
1    2726
dtype: int64

In [23]:
# Создаем таблицу сопряженности
h4_contingency = pd.crosstab(df_final['young_student'], df_final['target'])
print("Таблица сопряженности")
h4_contingency

Таблица сопряженности


target,Dropout,Graduate
young_student,,
0,584,320
1,837,1889


In [24]:
# Рассчитываем проценты
h4_percent = h4_contingency.div(h4_contingency.sum(axis=1), axis=0) * 100
print('Проценты (по строкам):')
h4_percent.round(1)

Проценты (по строкам):


target,Dropout,Graduate
young_student,,
0,64.6,35.4
1,30.7,69.3


In [25]:
# Тест хи-квадрат для проверки связи
# Рассчет Odds Ratio 

h4_result = calculate_chi2_or(h4_contingency)

Результаты теста:
хи-квадрат = 326.029
p-value = 0.00000000
Степени свободы (df) = 1
Статистически значимо
Есть связь между низким баллом и отчислением (p < 0.05)
------------------------
Отношение шансов:
Шанс отчисления в экспериментальной группе: 0.443 (1:2.3)
Шанс отчисления в контрольной группе: 1.825 (1:0.5)
Odds Ratio = 0.24
------------------------
Студенты в экспериментальной группе отчисляются реже (в 4.1 раз).
Гипотеза опровергнута.
Отчисляемость студентов:
Экспериментальная группа: 30.7% (837/2726)
Контрольная группа: 64.6% (584/904)


#### Гипотеза 4: Возраст студентов и отчисление


**Опровергнута**

*Результаты*
- Отчисляемость студентов до 25 лет: 30.7%
- Отчисляемость студентов старше 25 лет: 64.6%
- Студенты старше 25 лет отчисляются в **4.1 раз чаще** (OR=0.24, p<0.05)

*Вывод:* Студенты старше 25 лет отчисляются в 4.1 раз чаще студентов до 25 лет. 

*Возможные причины:*
1. Молодым людям проще адаптироваться к учебному процессу
2. Более взрослые студенты чаще совмещают учебу с работой и могут не справляться с нагрузкой
3. Мотивационный фактор — финансовые проблемы снижают фокус на учёбе

*Рекомендации:*
1. Признание опыта — зачёт профессиональных достижений и создание программ переподготовки вместо полного курса

2. Карьерная интеграция — проекты и стажировки, учитывающие имеющийся опыт и возрастные особенности

## Гипотеза 5: Низкий балл при поступлении увеличивает вероятность отчисления
**За низкий балл возьмем балл, ниже медианного**

In [26]:
# Рассчитываем медиану баллов при поступлении
median_admission = df_final['admission_grade'].median()
print(f"Медианный балл при поступлении: {median_admission:.2f}")

Медианный балл при поступлении: 126.50


In [27]:
# Создаем бинарную переменную: низкий балл (ниже медианы) vs высокий балл
df_final['low_admission'] = df_final['admission_grade'] < median_admission
df_final['low_admission_label'] = df_final['low_admission'].map({True: 'Низкий балл', False: 'Высокий балл'})

In [28]:
# Смотрим распределение
df_final.groupby('low_admission_label').size()

low_admission_label
Высокий балл    1821
Низкий балл     1809
dtype: int64

**Для проверки гипотезы используем статистический тест Хи-квадрат.**

In [29]:
# Создаем таблицу сопряженности
h5_contingency = pd.crosstab(df_final['low_admission_label'], df_final['target'])
print('Таблица сопряженности')
h5_contingency

Таблица сопряженности


target,Dropout,Graduate
low_admission_label,,
Высокий балл,631,1190
Низкий балл,790,1019


In [30]:
# Рассчитываем проценты
h5_percent = h5_contingency.div(h5_contingency.sum(axis=1), axis=0) * 100
print('Проценты (по строкам):')
h5_percent.round(1)

Проценты (по строкам):


target,Dropout,Graduate
low_admission_label,,
Высокий балл,34.7,65.3
Низкий балл,43.7,56.3


In [31]:
# Тест хи-квадрат для проверки связи
# Рассчет Odds Ratio h5_result = calculate_or(h5_contingency)
h5_result = calculate_chi2_or(h5_contingency)

Результаты теста:
хи-квадрат = 30.611
p-value = 0.00000003
Степени свободы (df) = 1
Статистически значимо
Есть связь между низким баллом и отчислением (p < 0.05)
------------------------
Отношение шансов:
Шанс отчисления в экспериментальной группе: 0.775 (1:1.3)
Шанс отчисления в контрольной группе: 0.530 (1:1.9)
Odds Ratio = 1.46
------------------------
Студенты в экспериментальной группе отчисляются в 1.5 раз чаще.
Гипотеза подтвердилась.
------------------------
Отчисляемость студентов:
Экспериментальная группа: 43.7% (790/1809)
Контрольная группа: 34.7% (631/1821)


#### Гипотеза 5: Влияние баллов при поступлении на отчисление

**Подтверждена**

*Результаты*
- Отчисляемость студентов с низким баллом: 43.7%
- Отчисляемость студентов с высоким баллом: 34.7%
- Студенты с низким баллом при поступлении отчисляются в **1.5 раз чаще** (OR=1.46, p<0.05)

*Вывод:* Студенты с баллом при поступлении ниже медианного отчисляются чаще в 1.5 раз чаще студентов с баллом выше медианного. 

*Возможные причины:* Студенты, имеющие при поступлении более низкие баллы, с большей вероятностью были хуже подготовлены к учебе и не справились с нагрузкой в вузе.
*Рекомендации*:
1. Программы адаптации — вводные интенсивные курсы перед началом учёбы для выравнивания базового уровня

2. Раннее кураторство — закрепление наставников с первого семестра для постоянного мониторинга прогресса и помощи

## Гипотеза 6: Наличие стипендии положительно влияет на успешное окончание обучения
**Для проверки гипотезы используем статистический тест Хи-квадрат.**

In [32]:
# Смотрим распределение
df_final.groupby('scholarship').size()

scholarship
0    2661
1     969
dtype: int64

In [33]:
# Создаем таблицу сопряженности
h6_contingency = pd.crosstab(df_final['scholarship'], df_final['target'])
print("Таблица сопряженности")

# Инверсия порядка для правильной работы функции (на 1 месте - контрольная группа)
h6_contingency = h6_contingency.iloc[::-1]  
h6_contingency

Таблица сопряженности


target,Dropout,Graduate
scholarship,,
1,134,835
0,1287,1374


In [34]:
# Рассчитываем проценты
h6_percent = h6_contingency.div(h6_contingency.sum(axis=1), axis=0) * 100
print('Проценты (по строкам):')
h6_percent.round(1)

Проценты (по строкам):


target,Dropout,Graduate
scholarship,,
1,13.8,86.2
0,48.4,51.6


In [35]:
# Тест хи-квадрат для проверки связи
# Рассчет Odds Ratio h5_result = calculate_or(h5_contingency)
h6_result = calculate_chi2_or(h6_contingency)

Результаты теста:
хи-квадрат = 354.219
p-value = 0.00000000
Степени свободы (df) = 1
Статистически значимо
Есть связь между низким баллом и отчислением (p < 0.05)
------------------------
Отношение шансов:
Шанс отчисления в экспериментальной группе: 0.937 (1:1.1)
Шанс отчисления в контрольной группе: 0.160 (1:6.2)
Odds Ratio = 5.84
------------------------
Студенты в экспериментальной группе отчисляются в 5.8 раз чаще.
Гипотеза подтвердилась.
------------------------
Отчисляемость студентов:
Экспериментальная группа: 48.4% (1287/2661)
Контрольная группа: 13.8% (134/969)


#### Гипотеза 6: Влияние стипендии на отчисление

**Подтверждена**

*Результаты*
- Отчисляемость студентов без стипендии: 48.4%
- Отчисляемость студентов со стипендией: 13.8%
- Студенты без стипендии отчисляются в **5.8 раз чаще** (OR=5.84, p<0.05)

*Вывод:* Студенты без стипендии отчисляются в 5.8 раз чаще студентов, получающих стипендию. 

*Возможные причины:* 
- Стипендию получают изначально более сильные студенты, она часто зависит от оценок, которые коррелируют с риском отчисления.
- Студенты без стипендии могут больше работать в ущерб учёбе.
- Необходимость поддерживать требования для стипендии развивает ответственность.
*Рекомендации*:
1. Расширение стипендиального охвата — ввести промежуточные виды поддержки для студентов с улучшающейся успеваемостью, также поощрять участие в исследованиях, конференциях и волонтёрских проектах

2. Гибкие критерии стипендии — учитывать не только оценки, но и прогресс, участие в проектах, сложные обстоятельства

3. Прозрачность и информирование — автоматические уведомления о приближении к стипендиальному порогу и консультации по его достижению

### Анализ по направлениям обучения

**Вопросы:**
- На каких специальностях самый высокий отток?
- Есть ли "проблемные" курсы, где больше 40% отчислений?

In [36]:
# Численный показатель оттока
dropouts_num = df[df['target'] == 'Dropout'].groupby('course').count()['target'].sort_values(ascending=False)
print('Число отчислившихся студентов по курсам')
dropouts_num

Число отчислившихся студентов по курсам


course
9991    136
9147    134
9500    118
9773    101
9254     96
9670     95
9119     92
9085     90
9003     86
9853     85
171      82
9130     78
8014     71
9238     65
9070     51
9556     33
33        8
Name: target, dtype: int64

In [37]:
# Отток студентов в процентах
dropout_percentages = (df[df['target'] == 'Dropout'].groupby('course').size() 
                       / df.groupby('course').size() * 100).round(1).sort_values(ascending=False)
print('Процент отчислившихся студентов по курсам')
dropout_percentages

Процент отчислившихся студентов по курсам


course
33      66.7
9130    55.3
9119    54.1
9991    50.7
9853    44.3
9003    41.0
9556    38.4
171     38.1
9254    38.1
9670    35.4
9147    35.3
8014    33.0
9773    30.5
9085    26.7
9070    22.6
9238    18.3
9500    15.4
dtype: float64

**Вывод:**

- Проблемные курсы с высоким процентом отчислений (>50%):
   1. Курс 33 - 66.7% отчислений (критический уровень)

   2. Курс 9130 - 55.3% отчислений (критический уровень)

   3. Курс 9119 - 54.1% отчислений (критический уровень)

   4. Курс 9991 - 50.7% отчислений (пороговый уровень)
 

- Курсы с наибольшим абсолютным числом отчислений:
 1. Курс 9991 - 136 студентов (самый высокий абсолютный показатель)

 2. Курс 9147 - 134 студентов

 3. Курс 9500 - 118 студентов
 

 

**Ключевые выводы:**

   1. Качественные проблемы:
   
     - Курсы 33, 9130, 9119 имеют системные проблемы (отчисляется больше половины студентов)

     - Курс 9991 требует особого внимания - 136 отчисленных при 50.7% оттока
     
   
   2. Количественные проблемы:
   
     - Курсы 9147 и 9500 - много отчисленных, но в разной пропорции

     - Курс 9147: 134 отчисленных при 35.3% оттока
 
     - Курс 9500: 118 отчисленных при всего 15.4% оттока
   

 **Рекомендации:**


   1. Срочные меры:
   
    - Аудит учебных программ курсов 33, 9130, 9119

    - Опрос отчисленных студентов с этих курсов
 
    - Пересмотр системы оценивания и требований
 


   2. Стратегические меры:
   
    - Анализ причин массовых отчислений на курсе 9991
    
    - Разработка системы раннего предупреждения для курсов с >40% оттока
    
    - Создание программ поддержки для проблемных курсов


# **Общие выводы и рекомендации по анализу студенческий успеваемости**


## **Ключевые выводы**

### **1. Доминирующий профиль успешного студента**
**Женщина-местная студентка** составляет самую многочисленную (1619 человек) и наиболее успешную группу, формируя позитивную статистику университета в целом. Эта группа демонстрирует наиболее высокие показатели выпуска и низкий процент отсева.

### **2. Проблемные группы**
**Мужчины-местные студенты** показывают наихудшие результаты с самым высоким процентом отчислений. Также повышенный риск отсева наблюдается у:
- Студентов старше 25 лет (отчисляются в 4.1 раза чаще)
- Вечерней формы обучения (в 1.7 раза чаще)
- Студентов с финансовыми трудностями (должники отчисляются в 5.9 раза чаще)

### **3. Критические академические факторы**
**Низкий средний балл** после первого семестра является самым сильным предиктором отчисления - студенты с баллом ниже медианы отчисляются в **7.6 раза чаще**. Это указывает на необходимость раннего вмешательства.

### **4. Проблемные учебные курсы**
Выявлены **4 критических курса** с уровнем отчислений более 50%:
- Курс 33: 66.7% отчислений (самый высокий процент)
- Курс 9130: 55.3% отчислений
- Курс 9119: 54.1% отчислений
- Курс 9991: 50.7% отчислений (136 студентов - наибольшее абсолютное число)



## **Три главные проблемы и рекомендации для их устранения**

### **Проблема 1: Низкий средний балл как главный предиктор отчисления**
**Факты:** Студенты с баллом ниже медианы отчисляются в 7.6 раза чаще

**Причины:** Недостаток базовых знаний, отсутствие мотивации, совмещение с работой

#### **Рекомендации:**

**1. Система раннего предупреждения (Early Warning System)**

- Внедрить автоматический мониторинг успеваемости после 1-го семестра
- Целевые показатели: балл ниже 60-го процентиля по курсу
- Автоматические алерт-уведомления для кураторов при выявлении риска


**2. Индивидуальные образовательные траектории**

- Разработать 3 уровня коррекционных программ:

    • Базовый (1-2 доп. занятия в неделю)
    
    • Интенсивный (летние/зимние школы)
    
    • Индивидуальный (тьюторское сопровождение)
    
    
- Обязательное тестирование для выявления пробелов


**3. Академическая поддержка как сервис**

- Создать Центр академического развития с услугами:

  • Коррекционные курсы по ключевым дисциплинам
  
  • Программа академического менторства
  
  • Тренинги по тайм-менеджменту и учебным навыкам


### **Проблема 2: Финансовые трудности как барьер для обучения**
**Факты:** Должники отчисляются в 5.9 раза чаще, студенты без стипендии - в 5.8 раза чаще

**Причины:** Финансовый стресс, необходимость работать в ущерб учебе, административные ограничения

#### **Рекомендации:**

**1. Гибкая система оплаты обучения**

- Внедрить 3 модели оплаты:
 
    • Рассрочка платежа по семестрам
 
    • Ступенчатая оплата (рост по мере обучения)
 
    • Success-based pricing (скидки за хорошую успеваемость)
  
- Автоматизация: интеграция с системой успеваемости


**2. Расширенная стипендиальная программа**

- Новые категории стипендий:

    • Стипендия за прогресс (рост успеваемости ≥15%)
  
    • Проектная стипендия (за участие в исследованиях)
  
    • Социальная стипендия (с учетом сложных обстоятельств)



**3. Финансовое консультирование и трудоустройство**

- Создать Службу финансовой поддержки студентов:
  
    • Консультации по бюджету и планированию
    
    • База проверенных подработок и стажировок
  
    • Программа work-study (работа в университете)
 
 
- Партнерства с компаниями для гибкого графика


### **Проблема 3: Системные проблемы конкретных курсов**
**Факты:** 4 курса имеют уровень отчислений >50%, курс 9991 - 136 отчисленных студентов
**Причины:** Сложность программ, неадекватные требования, недостаток поддержки

#### **Рекомендации:**

**1. Экстренный аудит проблемных курсов**

- Независимая комиссия для курсов 33, 9130, 9119, 9991:
 
    • Анализ учебных программ и требований
 
    • Опрос отчисленных и текущих студентов
 
    • Бенчмаркинг с аналогичными курсами других вузов


**2. Редизайн учебных программ**

- Для каждого проблемного курса:
  
    • Модуляризация (разбивка на smaller chunks)
  
    • Добавление промежуточных контрольных точек
  
    • Введение факультативной поддержки
  
    • Пересмотр системы оценивания


**3. Программа поддержки преподавателей**

- Материальное стимулирование за снижение отсева

- Тренинги и ресурсы для преподавателей проблемных курсов:
  
    • Методики работы со слабыми студентами
  
    • Инструменты формирующего оценивания
  
    • Коучинг от успешных преподавателей
  
 


## **Стратегические инициативы**

### **Инициатива "Студент первого года"**

Цель: Снизить отсев на 25% за 2 года

Компоненты:

1. Вводный интенсивный модуль (2 недели до начала учёбы)
2. Обязательное закрепление академического наставника
3. Еженедельный мониторинг прогресса в 1-м семестре
4. Портфолио студента с индивидуальным планом развития


### **Цифровая платформа академической аналитики**

Цель: Прогнозирование рисков отсева с точностью 85%

Функционал:

  
    • Real-time дашборды для деканатов
  
    • Predictive модели риска отчисления
  
    • Автоматические рекомендации действий
  
    • Интеграция с существующими системами

Технологии: ML-модели, API интеграция


### **Партнерская программа с работодателями**

Цель: Снизить финансовое давление через трудоустройство

Модели:

1. Corporate scholarships (стипендии от компаний)

2. Guaranteed internships (гарантированные стажировки)

3. Flexible work programs (гибкий график под учебу)



## **ПРИОРИТЕТЫ РЕАЛИЗАЦИИ**

### **Срочные (0-3 месяца):**

1. Аудит курсов 33, 9130, 9119, 9991

2. Внедрение системы мониторинга после 1-го семестра

3. Расширение стипендиального фонда

### **Среднесрочные (3-12 месяцев):**

1. Запуск Центра академического развития

2. Внедрение гибкой системы оплаты

3. Редизайн учебных программ проблемных курсов

### **Долгосрочные (12-36 месяцев):**

1. Цифровая платформа академической аналитики

2. Полная персонализация образовательных траекторий

3. Устойчивое партнерство с работодателями

## **ЗАКЛЮЧЕНИЕ**

Анализ выявил четкие паттерны успеха и отсева, позволяющие перейти от реактивного к проактивному управлению студенческим контингентом. **Финансовые трудности, низкая начальная успеваемость и системные проблемы конкретных курсов** требуют немедленного внимания и комплексного решения.

Предлагаемые меры сочетают **технологические инновации** (прогностическая аналитика), **организационные изменения** (гибкие системы оплаты и поддержки) и **образовательные трансформации** (редизайн курсов, персонализация обучения). Реализация этих рекомендаций позволит не только снизить отсев, но и повысить общее качество образовательного процесса, укрепив позиции университета на рынке высшего образования.

**Ключевой принцип:** Переход от модели "выживания сильнейших" к модели "успеха каждого студента" через своевременную поддержку и персонализацию образовательного пути.